# 03_spatial_reconstruction_analysis.ipynb

Análisis de calidad de reconstrucción espacial del autoencoder para Valle de Aconcagua

Este notebook genera visualizaciones de la calidad de reconstrucción del AE/VAE, enfocado en:
- Métricas globales de desempeño (MSE, R², Silhouette)
- Error de reconstrucción disagregado por variable climática
- Patrones espaciales de error (heterogeneidad por zona geográfica)
- Distribución del error por escenario SSP (SSP245, SSP370, SSP585)

**Hipótesis**: La reconstrucción espacial del autoencoder debe presentar heterogeneidad territorial correlacionada con factores climáticos y topográficos, reflejando resilencia diferencial en zonas del valle.


In [11]:
import os
import pickle
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.stats import percentileofscore
from scipy.special import softmax
import warnings
warnings.filterwarnings('ignore')
import sys
from pathlib import Path

MODEL_ORDER = ["AE", "VAE"]

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


BASE_DIR = "/home/aninotna/magister/tesis/justh2_pipeline"
SCRIPTS_DIR = os.path.join(BASE_DIR, "scripts/idroverdi_autoencoder_3")
PLOTS_DIR = Path(BASE_DIR) / "plots" / "autoencoder_analysis"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, SCRIPTS_DIR)

print(f"Base directory: {BASE_DIR}")
print(f"Plots directory: {PLOTS_DIR}")
print(f"\nLibraries imported successfully")

Base directory: /home/aninotna/magister/tesis/justh2_pipeline
Plots directory: /home/aninotna/magister/tesis/justh2_pipeline/plots/autoencoder_analysis

Libraries imported successfully


## Section 1: Load Data from 07_experiments_1_clustering_1511

Cargamos los datos de reconstrucción del notebook de experimentos principal. Los datos incluyen:
- Datos originales normalizados (X_norm) y datos originales sin normalizar (X_orig) para cada SSP
- Datos reconstruidos por el autoencoder
- Coordenadas geográficas (lat/lon) para cada píxel
- Nombres de variables climáticas


In [12]:
class AE(nn.Module):
    def __init__(self, input_dim, latent_dim=8, p_drop=0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Linear(64, 128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Linear(128, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z


class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=12, p_drop=0.05):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.1, inplace=True),
        )
        self.mu = nn.Linear(64, latent_dim)
        self.logvar = nn.Linear(64, latent_dim)

        self.dec = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Linear(64, 128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Linear(256, input_dim),
        )
    
    def encode(self, x):
        h = self.enc(x)
        return self.mu(h), self.logvar(h)
    
    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        x_hat = self.dec(z)
        return x_hat, mu, logvar

print("Arquitecturas AE y VAE definidas")

Arquitecturas AE y VAE definidas


In [14]:
print("CARGANDO DATOS DESDE EXPERIMENTO 1")
print()

# Buscar el archivo más reciente automáticamente
import glob
DATA_DIR = os.path.join(BASE_DIR, "data")
trained_dir = os.path.join(DATA_DIR, "autoencoder_trained_v2")
pattern = os.path.join(trained_dir, "experiment1_clustering_*.pkl")
available_files = glob.glob(pattern)

if not available_files:
    raise FileNotFoundError(f"No se encontraron archivos en: {pattern}")

# Ordenar por fecha de modificación (más reciente primero)
export_path = max(available_files, key=os.path.getmtime)

print(f"Archivo más reciente encontrado: {os.path.basename(export_path)}")
print(f"Ruta completa: {export_path}")
print()

if os.path.exists(export_path):
    print(f"Cargando datos...")
    print()
    
    with open(export_path, "rb") as f:
        exp1_data = pickle.load(f)
    
    # 1. Cargar modelos PyTorch
    print("1. Cargando modelos PyTorch...")
    models_path = exp1_data["models_path"]
    model_dims = exp1_data["model_dims"]
    
    MODELS = {}
    for model_key, model_file in models_path.items():
        dims = model_dims[model_key]
        
        if "AE" in model_key and "VAE" not in model_key:
            model = AE(
                input_dim=dims["input_dim"],
                latent_dim=dims["latent_dim"]
            )
        else:
            model = VAE(
                input_dim=dims["input_dim"],
                latent_dim=dims["latent_dim"]
            )
        
        model.load_state_dict(torch.load(model_file))
        model.eval()
        MODELS[model_key] = model
        print(f"  ✓ {model_key} (input={dims['input_dim']}, latent={dims['latent_dim']})")
    print()
    
    # 2. Extraer objetos del pickle
    print("2. Cargando datos adicionales...")
    LATENTS = exp1_data["LATENTS"]
    LATENT_LOGVARS = exp1_data.get("LATENT_LOGVARS", None)
    
    MODEL_ORDER = exp1_data["MODEL_ORDER"]
    LATENT_DIM_AE = exp1_data["LATENT_DIM_AE"]
    LATENT_DIM_VAE = exp1_data["LATENT_DIM_VAE"]
    N_PER_SCENARIO = exp1_data["N_PER_SCENARIO"]
    
    X_BASE = exp1_data["X_BASE"]
    X245_orig = exp1_data["X245_orig"]
    X370_orig = exp1_data["X370_orig"]
    X585_orig = exp1_data["X585_orig"]
    
    X245_norm = exp1_data.get("X245_norm", None)
    X370_norm = exp1_data.get("X370_norm", None)
    X585_norm = exp1_data.get("X585_norm", None)
    
    feature_names = exp1_data["feature_names"]
    coords_df = exp1_data["coords_df"]
    
    file_size_mb = os.path.getsize(export_path) / (1024 * 1024)
    
    print(f"  ✓ Datos cargados ({file_size_mb:.2f} MB)")
    print()
    
    print("Objetos cargados:")
    print(f"  • MODELS: {len(MODELS)} modelos ({', '.join(MODELS.keys())})")
    print(f"  • LATENTS: {len(LATENTS)} conjuntos de embeddings")
    print(f"  • N_PER_SCENARIO: {N_PER_SCENARIO} puntos espaciales")
    print(f"  • feature_names: {len(feature_names)} variables")
    print(f"  • coords_df: {coords_df.shape[0]} píxeles")
    print()
    
else:
    raise FileNotFoundError(f"No se encontró {export_path}")

CARGANDO DATOS DESDE EXPERIMENTO 1

Archivo más reciente encontrado: experiment1_clustering_20251209_001834.pkl
Ruta completa: /home/aninotna/magister/tesis/justh2_pipeline/data/autoencoder_trained_v2/experiment1_clustering_20251209_001834.pkl

Cargando datos...

1. Cargando modelos PyTorch...
  ✓ AE (input=29, latent=8)
  ✓ VAE (input=29, latent=8)

2. Cargando datos adicionales...
  ✓ Datos cargados (2.00 MB)

Objetos cargados:
  • MODELS: 2 modelos (AE, VAE)
  • LATENTS: 2 conjuntos de embeddings
  • N_PER_SCENARIO: 661 puntos espaciales
  • feature_names: 47 variables
  • coords_df: 661 píxeles

